# Séries temporelles avec pandas : la station météo d'Iéna

La station météo de l'Institut Max-Planck de biogéochimie, à Iéna (Allemagne), mesure 14 grandeurs toutes les 10 minutes : température, pression, humidité, vent… On en a ici huit ans, de 2009 à 2016, soit 420 000 lignes : de quoi contrôler un index temporel, nettoyer une série, puis la résumer par jour, par mois et par saison.

1. Lire le fichier
2. Contrôler l'index temporel
3. Corriger l'index
4. Valeurs aberrantes du vent
5. Rééchantillonner par jour et par mois
6. Normales et anomalies mensuelles
7. Moyenne glissante et saisonnalité

Chaque partie part du résultat de la précédente. Le notebook tourne sans GPU, en une minute environ.

Les données viennent de la [station de l'Institut Max-Planck de biogéochimie](https://www.bgc-jena.mpg.de/wetter/), dans la version qu'en a préparée le [tutoriel TensorFlow sur les séries temporelles](https://www.tensorflow.org/tutorials/structured_data/time_series) (code sous licence Apache 2.0), d'où viennent aussi la correction du vent de la partie 4 et la remarque sur la direction du vent de la partie 5.

## Données

Une ligne par mesure, horodatée par la colonne `Date Time` au format `jour.mois.année heure:minute:seconde`. Les colonnes : `p (mbar)` la pression, `T (degC)` la température, `Tpot (K)` la température potentielle, `Tdew (degC)` le point de rosée, `rh (%)` l'humidité relative, `VPmax (mbar)`, `VPact (mbar)` et `VPdef (mbar)` les pressions de vapeur saturante et réelle, et leur différence, `sh (g/kg)` l'humidité spécifique, `H2OC (mmol/mol)` la concentration en vapeur d'eau, `rho (g/m**3)` la masse volumique de l'air, `wv (m/s)` la vitesse moyenne du vent, `max. wv (m/s)` sa vitesse maximale (les rafales) et `wd (deg)` sa direction, en degrés (0 : vent du nord, 90 : vent d'est).

In [ ]:
!wget -q -nc https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## 1. Lire le fichier

Lisez `jena_climate_2009_2016.csv.zip` dans la `DataFrame` `df` avec `pd.read_csv`, sans le décompresser : la colonne `Date Time` en index, convertie en dates dès la lecture, avec un format explicite. Affichez le type de l'index, qui doit être `datetime64[ns]` et non `object`, et les premières lignes. Pourquoi donner le format plutôt que de laisser pandas le deviner ?

**Documentation utile**

- [`pd.read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les paramètres de `pd.read_csv` :

- `index_col="Date Time"` : la colonne qui devient l'index
- `parse_dates=True` : convertit l'index en dates
- `date_format="%d.%m.%Y %H:%M:%S"` : le format de `01.01.2009 00:10:00` (jour, mois, année, heure, minute, seconde)
- pandas décompresse lui-même un `.zip` qui contient un seul fichier
- `df.index.dtype` : le type de l'index

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
df = pd.read_csv(
  "jena_climate_2009_2016.csv.zip",
  index_col=...,
  parse_dates=True,
  date_format=...,
)
print(f"Type de l'index : {df.index.dtype}")
df.head()
```

### Solution

In [ ]:
df = pd.read_csv(
  "jena_climate_2009_2016.csv.zip",
  index_col="Date Time",
  parse_dates=True,
  date_format="%d.%m.%Y %H:%M:%S",
)
print(f"Type de l'index : {df.index.dtype}")
# Sans format, pandas devine « mois d'abord » sur 01.01.2009, échoue dès le
# 13.01.2009 et laisse l'index en chaînes (type object), sans erreur ni avertissement.
df.head()

## 2. Contrôler l'index temporel

Avant tout calcul sur une série temporelle, son index doit être trié, sans doublon, et à pas régulier. Vérifiez-le sur `df` en affichant :

- la forme de `df` et la période couverte (première et dernière date) ;
- si l'index est trié ;
- combien d'horodatages apparaissent plusieurs fois, et combien de lignes sont entièrement dupliquées, horodatage compris ;
- combien de fois apparaît chaque écart entre deux horodatages consécutifs ;
- `df.describe().T`.

Le pas attendu est de 10 minutes : que signifient les écarts négatifs ? Quelles valeurs de `describe` sont impossibles ? (Vous les corrigerez à la partie 4.)

**Documentation utile**

- [`Index.is_monotonic_increasing`](https://pandas.pydata.org/docs/reference/api/pandas.Index.is_monotonic_increasing.html)
- [`Index.duplicated`](https://pandas.pydata.org/docs/reference/api/pandas.Index.duplicated.html)
- [`DataFrame.duplicated`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html)
- [`Index.to_series`](https://pandas.pydata.org/docs/reference/api/pandas.Index.to_series.html)
- [`Series.diff`](https://pandas.pydata.org/docs/reference/api/pandas.Series.diff.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels à utiliser :

- `df.index.min()` et `df.index.max()` : la première et la dernière date
- `df.index.is_monotonic_increasing` : `True` si l'index est trié par ordre croissant
- `df.index.duplicated()` : un booléen par ligne, `True` quand l'horodatage a déjà été vu plus haut ; `.sum()` compte les `True`
- `df.duplicated()` compare les valeurs des colonnes, sans l'index : sur `df.reset_index()`, l'horodatage redevient une colonne, et la comparaison porte sur la ligne entière
- `df.index.to_series().diff()` : l'écart entre chaque horodatage et le précédent (un `Timedelta`) ; `.value_counts()` compte chaque écart

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print(f"Forme : {df.shape}")
print(f"Période : {df.index.min()} → {...}")
print(f"Index trié : {...}")
print(f"Horodatages en double : {df.index.duplicated().sum()}")
print(f"Lignes entièrement dupliquées : {...}")
print(df.index.to_series().diff().value_counts())
df.describe().T
```

### Solution

In [ ]:
print(f"Forme : {df.shape}")
print(f"Période : {df.index.min()} → {df.index.max()}")
print(f"Index trié : {df.index.is_monotonic_increasing}")
print(f"Horodatages en double : {df.index.duplicated().sum()}")
print(f"Lignes entièrement dupliquées : {df.reset_index().duplicated().sum()}")
print(df.index.to_series().diff().value_counts())
# Deux écarts négatifs : l'index recule deux fois (de 23 h 50, puis de 30 h 20) et
# repasse sur des horodatages déjà vus : ce sont les 327 lignes en double, identiques.
# describe : wv (m/s) et max. wv (m/s) descendent à -9999, une vitesse impossible.
df.describe().T

## 3. Corriger l'index

Les lignes en double sont identiques : gardez la première ligne de chaque horodatage, triez l'index et réaffectez le résultat à `df`. Vérifiez que l'index est maintenant trié et sans doublon, puis comptez à nouveau les écarts entre horodatages consécutifs, dans `pas`. Affichez enfin les écarts de plus de 10 minutes : combien de temps a duré la plus longue interruption de la station, et quand ?

**Documentation utile**

- [`DataFrame.sort_index`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_index.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels à utiliser :

- `~df.index.duplicated()` : `True` pour la première apparition de chaque horodatage (`~` inverse un masque booléen) ; `df[masque]` garde les lignes où le masque vaut `True`
- `.sort_index()` : trie les lignes selon l'index
- `pas > pd.Timedelta("10min")` : `True` pour les écarts de plus de 10 minutes ; `pas[...]` les garde, chacun daté par la fin de l'interruption

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
df = df[...].sort_index()
print(f"Index trié : {df.index.is_monotonic_increasing}")
print(f"Horodatages en double : {...}")

pas = df.index.to_series().diff()
print(pas.value_counts())
pas[pas > ...]
```

### Solution

In [ ]:
df = df[~df.index.duplicated()].sort_index()
print(f"Index trié : {df.index.is_monotonic_increasing}")
print(f"Horodatages en double : {df.index.duplicated().sum()}")

pas = df.index.to_series().diff()
print(pas.value_counts())
# La plus longue interruption, 3 jours et 2 h 20, s'achève le 28/10/2016 à 12 h 50 :
# les 26 et 27 octobre 2016 n'ont aucune mesure.
pas[pas > pd.Timedelta("10min")]

## 4. Valeurs aberrantes du vent

`describe` l'a montré : `wv (m/s)` et `max. wv (m/s)` valent parfois `-9999.0`, un code d'erreur du capteur et non une vitesse. Le remplacer par `0` inventerait des moments sans vent : remplacez-le par `NaN` dans ces deux colonnes et comptez les valeurs manquantes obtenues. Comblez-les ensuite par une interpolation selon le temps : la valeur, sur la droite qui joint la mesure d'avant et celle d'après, à la date du trou. Vérifiez enfin avec `describe` que les deux colonnes n'ont plus ni valeur négative ni valeur manquante.

**Documentation utile**

- [`DataFrame.replace`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.replace.html)
- [`DataFrame.interpolate`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.interpolate.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `vent = ["wv (m/s)", "max. wv (m/s)"]` : `df[vent]` sélectionne les deux colonnes, et `df[vent] = ...` les remplace
- `.replace(-9999.0, np.nan)` : renvoie une copie modifiée, à réaffecter. N'extrayez pas une colonne pour la modifier à part (`wv = df["wv (m/s)"]`, puis `wv[masque] = ...`) : cela modifie `df` en pandas 2, mais plus en pandas 3, avec le Copy-on-Write
- `.isna().sum()` : le nombre de valeurs manquantes de chaque colonne
- `.interpolate(method="time")` : interpolation linéaire selon les dates de l'index, et non selon la position des lignes (les deux diffèrent là où la station s'est arrêtée)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
vent = ["wv (m/s)", "max. wv (m/s)"]
df[vent] = df[vent].replace(..., np.nan)
print(df[vent].isna().sum())
df[vent] = df[vent].interpolate(method=...)
df[vent].describe()
```

### Solution

In [ ]:
vent = ["wv (m/s)", "max. wv (m/s)"]
df[vent] = df[vent].replace(-9999.0, np.nan)
print(df[vent].isna().sum())
df[vent] = df[vent].interpolate(method="time")
df[vent].describe()

## 5. Rééchantillonner par jour et par mois

Une mesure toutes les 10 minutes, c'est trop fin pour parler de climat. Gardez d'abord les seules mesures de 2009 à 2016 : la dernière, le 1er janvier 2017 à 0 h, serait seule dans son jour. Rééchantillonnez ensuite `df` par jour dans `quotidien`, avec une agrégation par colonne : les températures moyenne, minimale et maximale (`t_moy`, `t_min`, `t_max`), la pression moyenne (`p_moy`), la vitesse moyenne du vent (`vent_moy`) et la plus forte rafale (`rafale_max`). Quels jours n'ont aucune mesure ? Quels jours ont connu le plus grand froid, la plus forte chaleur, la plus forte rafale ?

Rééchantillonnez enfin `quotidien` par mois dans `mensuel`, chaque colonne avec son agrégation : moyenne des moyennes, minimum des minimums, maximum des maximums.

La direction du vent, `wd (deg)`, reste de côté : la moyenne de 350° et de 10°, deux vents du nord, donne 180°, un vent du sud. Une direction se moyenne par ses composantes (`np.cos` et `np.sin` de l'angle, multipliés par la vitesse), comme dans le tutoriel TensorFlow.

**Documentation utile**

- [`DataFrame.resample`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html)
- [`Resampler.aggregate`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.Resampler.aggregate.html)
- [`Series.idxmax`](https://pandas.pydata.org/docs/reference/api/pandas.Series.idxmax.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `df.loc[:"2016"]` : sélection partielle d'un index de dates, jusqu'à la fin de 2016 incluse
- `df.resample("1D")` regroupe les lignes par jour ; `.agg(nom=("colonne", "fonction"), ...)` crée une colonne `nom` par agrégation (`"mean"`, `"min"`, `"max"`)
- un jour sans mesure donne une ligne de `NaN` : `quotidien.isna().all(axis=1)` les repère, `quotidien.index[...]` donne leurs dates
- `quotidien["t_min"].idxmin()` : la date du minimum, `idxmax()` celle du maximum ; `quotidien.loc[liste_de_dates]` affiche ces jours
- par mois : `.resample("ME")` (`ME` pour *month end* : chaque mois est daté de son dernier jour), puis `.agg({"colonne": "fonction", ...})`, une fonction par colonne

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
df = df.loc[...]
quotidien = df.resample(...).agg(
  t_moy=("T (degC)", "mean"),
  t_min=("T (degC)", ...),
  t_max=...,
  p_moy=...,
  vent_moy=...,
  rafale_max=...,
)
vides = quotidien.index[quotidien.isna().all(axis=1)]
print("Jours sans mesure :", list(vides.strftime("%Y-%m-%d")))
jours = [
  quotidien["t_min"].idxmin(),
  ...,  # la date de la plus forte chaleur
  ...,  # celle de la plus forte rafale
]
print(quotidien.loc[jours].round(1))

mensuel = quotidien.resample(...).agg(
  {
    "t_moy": "mean",
    "t_min": ...,
    "t_max": ...,
    "p_moy": ...,
    "vent_moy": ...,
    "rafale_max": ...,
  }
)
mensuel.round(1)
```

### Solution

In [ ]:
df = df.loc[:"2016"]
quotidien = df.resample("1D").agg(
  t_moy=("T (degC)", "mean"),
  t_min=("T (degC)", "min"),
  t_max=("T (degC)", "max"),
  p_moy=("p (mbar)", "mean"),
  vent_moy=("wv (m/s)", "mean"),
  rafale_max=("max. wv (m/s)", "max"),
)
vides = quotidien.index[quotidien.isna().all(axis=1)]
print("Jours sans mesure :", list(vides.strftime("%Y-%m-%d")))
jours = [
  quotidien["t_min"].idxmin(),
  quotidien["t_max"].idxmax(),
  quotidien["rafale_max"].idxmax(),
]
print(quotidien.loc[jours].round(1))
# Plus grand froid : -23,0 °C le 07/01/2009 ; plus forte chaleur : 37,3 °C le
# 04/07/2015 ; plus forte rafale : 23,5 m/s (85 km/h) le 28/02/2010, la tempête Xynthia.

mensuel = quotidien.resample("ME").agg(
  {
    "t_moy": "mean",
    "t_min": "min",
    "t_max": "max",
    "p_moy": "mean",
    "vent_moy": "mean",
    "rafale_max": "max",
  }
)
mensuel.round(1)

## 6. Normales et anomalies mensuelles

La normale d'un mois est sa température moyenne sur plusieurs années (30 ans pour une normale officielle, 8 ici), et l'anomalie d'un mois donné, son écart à cette normale. Avec `pivot_table`, faites de la colonne `t_moy` de `mensuel` un tableau `table` d'une ligne par année et d'une colonne par mois (1 à 12) : `table.loc[2009, 1]`, janvier 2009, vaut -3,6 °C environ. Affichez ensuite :

- le mois le plus chaud de chaque année ;
- pour chaque mois, l'année où il a été le plus chaud par rapport à sa normale (`normale` : la normale de chaque mois sur 2009-2016 ; `anomalie` : le tableau des anomalies) ;
- l'anomalie moyenne de chaque année : quelle année a été la plus chaude ?
- `anomalie` en carte de chaleur, centrée sur 0.

**Documentation utile**

- [`DataFrame.pivot_table`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html)
- [`DataFrame.idxmax`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.idxmax.html)
- [`sns.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `mensuel.assign(annee=mensuel.index.year, mois=mensuel.index.month)` : ajoute l'année et le mois de chaque ligne en colonnes
- `.pivot_table(index="annee", columns="mois", values="t_moy")` : une ligne par année, une colonne par mois
- `table.idxmax(axis=1)` : pour chaque ligne, la colonne du maximum (le mois) ; sans `axis`, pour chaque colonne, la ligne du maximum (l'année)
- `table.mean()` : la moyenne de chaque colonne, donc de chaque mois ; `table - normale` soustrait à chaque colonne sa normale (pandas aligne l'index de la `Series` sur les colonnes du tableau)
- `anomalie.mean(axis=1)` : la moyenne de chaque ligne, donc de chaque année
- `sns.heatmap(anomalie, center=0, cmap="coolwarm", annot=True, fmt=".1f")` : en bleu sous la normale, en rouge au-dessus, la valeur dans chaque case

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
table = mensuel.assign(annee=mensuel.index.year, mois=mensuel.index.month).pivot_table(
  index=..., columns=..., values="t_moy"
)
print(f"Mois le plus chaud de chaque année :\n{table.idxmax(axis=...)}")

normale = ...
anomalie = table - ...
print(
  f"Année la plus chaude de chaque mois, par rapport à la normale :\n{anomalie.idxmax()}"
)
print(f"Anomalie moyenne de chaque année :\n{anomalie.mean(axis=...).round(2)}")
sns.heatmap(anomalie, center=..., cmap="coolwarm", annot=True, fmt=".1f")
plt.show()
```

### Solution

In [ ]:
table = mensuel.assign(annee=mensuel.index.year, mois=mensuel.index.month).pivot_table(
  index="annee", columns="mois", values="t_moy"
)
print(f"Mois le plus chaud de chaque année :\n{table.idxmax(axis=1)}")

normale = table.mean()
anomalie = table - normale
print(
  f"Année la plus chaude de chaque mois, par rapport à la normale :\n{anomalie.idxmax()}"
)
print(f"Anomalie moyenne de chaque année :\n{anomalie.mean(axis=1).round(2)}")
# Juillet ou août selon les années ; 2010 est la plus froide (-2,0 °C), 2014 la plus
# chaude (+1,3 °C), devant 2015.
sns.heatmap(anomalie, center=0, cmap="coolwarm", annot=True, fmt=".1f")
plt.show()

## 7. Moyenne glissante et saisonnalité

Calculez `lissee`, la moyenne glissante sur 30 jours de la température moyenne de chaque jour (`quotidien["t_moy"]`), puis tracez sur un même graphique la série et `lissee`, pour la seule année 2012. Regardez la vague de froid de février : la courbe lissée est-elle en avance ou en retard sur la série ? Pourquoi ? Tracez ensuite une boîte à moustaches de `t_moy` par mois : quels mois varient le plus d'un jour à l'autre ?

**Documentation utile**

- [`Series.rolling`](https://pandas.pydata.org/docs/reference/api/pandas.Series.rolling.html)
- [`DataFrame.plot.box`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.box.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `.rolling("30D").mean()` : pour chaque jour, la moyenne des 30 jours qui finissent ce jour-là. Calculez-la sur toute la série avant de garder 2012 : sinon, les 29 premiers jours de 2012 n'ont qu'une fenêtre partielle
- `quotidien.loc["2012", "t_moy"]` et `lissee.loc["2012"]` : l'année 2012 seule
- `serie.plot(figsize=(12, 4), alpha=0.5, label="Moyenne du jour")` : renvoie les axes du graphique (`ax`) ; un second `.plot(ax=ax, label=...)` trace sur les mêmes axes, et `ax.legend()` affiche la légende
- `quotidien.assign(mois=quotidien.index.month)` ajoute le mois en colonne ; `.plot.box(column="t_moy", by="mois", figsize=(10, 4))` : une boîte par mois
- `plt.show()` après chaque graphique

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
lissee = quotidien["t_moy"].rolling(...).mean()
ax = quotidien.loc["2012", "t_moy"].plot(
  figsize=(12, 4), alpha=0.5, label="Moyenne du jour"
)
lissee.loc[...].plot(ax=..., label="Moyenne glissante sur 30 jours")
ax.legend()
plt.show()

quotidien.assign(mois=...).plot.box(column="t_moy", by=..., figsize=(10, 4))
plt.show()
```

### Solution

In [ ]:
lissee = quotidien["t_moy"].rolling("30D").mean()
ax = quotidien.loc["2012", "t_moy"].plot(
  figsize=(12, 4), alpha=0.5, label="Moyenne du jour"
)
lissee.loc["2012"].plot(ax=ax, label="Moyenne glissante sur 30 jours")
ax.legend()
plt.show()

# En retard : la vague de froid de février 2012 atteint son minimum le 6, la moyenne
# glissante le 13. Celle d'un jour porte sur les 30 jours qui finissent ce jour-là
# (center=True centre la fenêtre). Les mois d'hiver varient le plus : un écart-type
# de 5,1 °C en janvier et février, contre 3,3 °C en août.
quotidien.assign(mois=quotidien.index.month).plot.box(
  column="t_moy", by="mois", figsize=(10, 4)
)
plt.show()